# Démarrage rapide (français) avec `privacy_service`

Ce notebook montre comment utiliser la bibliothèque **Privacy Service** pour :

- détecter des informations personnelles (PII) dans des textes **en français** ;
- anonymiser ces informations selon différentes stratégies (remplacement, masquage, etc.) ;
- utiliser les **modèles personnalisés** prévus pour des numéros de bénévoles et de dossiers.

L’objectif est que vous puissiez rapidement vérifier que la librairie fonctionne dans votre environnement, sans avoir à lire tout le code.


## 1. Préparer la configuration

Avant de commencer, assurez-vous d’avoir un fichier de configuration adapté.

Dans ce dépôt, vous pouvez partir de :

- `config.example.yaml` (exemple complet, déjà orienté français) ;
- ou d’un fichier `config.yaml` que vous aurez copié/ajusté à partir de l’exemple.

Dans les exemples ci‑dessous, on privilégie une configuration **en français** (`language: fr`) et l’usage d’**AI4Privacy** pour la détection sémantique.


In [2]:
from pathlib import Path

from privacy_service import PrivacyService

# On cherche d'abord un config.yaml local, sinon on tombe sur config.example.yaml
config_path = None
for candidate in (Path("config.yaml"), Path("config.example.yaml")):
    if candidate.exists():
        config_path = candidate
        break

if config_path is None:
    print(
        "⚠ Aucun fichier de configuration trouvé. Utilisation des valeurs par défaut (anglais)."
    )
    service = PrivacyService()
else:
    print(f"✓ Fichier de configuration trouvé : {config_path}")
    service = PrivacyService(config=str(config_path))

print("\nConfiguration active :")
print(f"  Langue                               : {service.config.language}")
print(f"  AI4Privacy activé                    : {service.config.use_ai4privacy}")
print(
    f"  Reconnaisseurs par défaut Presidio   : {service.config.use_presidio_defaults}"
)
print(f"  Reconnaisseurs spaCy activés         : {service.config.use_spacy_nlp}")
print(
    f"  Stratégie d'anonymisation par défaut : {service.config.default_anonymization_strategy}"
)

✓ Fichier de configuration trouvé : config.example.yaml


/home/gregoire/silex/privacy-service/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Configuration active :
  Langue                               : fr
  AI4Privacy activé                    : True
  Reconnaisseurs par défaut Presidio   : True
  Reconnaisseurs spaCy activés         : True
  Stratégie d'anonymisation par défaut : replace


## 2. Détecter des PII dans un texte français simple

On commence par un exemple basique : coordonnées d’un·e bénéficiaire en français.
Nous allons uniquement **détecter** les entités pour vérifier que la configuration fonctionne.


In [3]:
texte_fr = """
Ma mère Astrit Nani Kofi est née à Ruswil en février/75
"""

print("Texte original :")
print(texte_fr)
print("\n--- Détections ---")

# Si votre config est en français, vous pouvez omettre language="fr"
detections = service.detect(texte_fr, language="fr")
for det in detections:
    print(
        f"{det.entity_type:20s} | {det.text:30s} | score={det.score:.3f} | via={det.recognizer}"
    )

Texte original :

Ma mère Astrit Nani Kofi est née à Ruswil en février/75


--- Détections ---

[ai4privacy] Disclaimer 📢:
AI4Privacy is trained on the world's largest open-source privacy dataset. 
For production use, please evaluate results carefully. For assistance, contact
us at our website https://ai4privacy.com or email support@ai4privacy.com.


[ai4privacy] Loading model: ai4privacy/llama-ai4privacy-multilingual-categorical-anonymiser-openpii...
PERSON               | Astrit Nani Kofi               | score=0.850 | via=SpacyRecognizer
LOCATION             | Ruswil                         | score=0.850 | via=SpacyRecognizer
DATE_TIME            |  février/75
                   | score=0.517 | via=AI4PrivacyRecognizer
LOCATION             |  Ruswil                        | score=0.002 | via=AI4PrivacyRecognizer
GENDER               |  mère                          | score=0.001 | via=AI4PrivacyRecognizer
PERSON               |  Astrit                        | score=0.000 | via=AI4Pr

## 3. Anonymiser un texte français

Nous allons maintenant appliquer différentes stratégies d’anonymisation sur un texte en français :

- `replace` : remplace par un placeholder comme `[EMAIL_ADDRESS]` ;
- `mask` : masque le texte par des `*` ;
- `redact` : supprime complètement ;
- `hash` : remplace par un hash irréversible.


In [3]:
texte_fr2 = """
Coordonnées du contact :
- Nom : Jean Martin
- Email : jean.martin@example.fr
- Téléphone : +33 6 12 34 56 78
- Adresse : 15 Rue de la Paix, 75001 Paris
"""

print("Texte original :")
print(texte_fr2)

strategies = ["replace", "mask", "redact", "hash"]

for strat in strategies:
    print("\n" + "=" * 60)
    print(f"Stratégie : {strat}")
    resultat = service.anonymize(texte_fr2, language="fr", strategy=strat)
    print(resultat.text)
    print(f"Anonymisations : {len(resultat.items)}")
    for item in resultat.items:
        print(
            f"  - {item.entity_type:20s} : '{item.text}' → '{item.anonymized_text}' (op={item.operator})"
        )

Texte original :

Coordonnées du contact :
- Nom : Jean Martin
- Email : jean.martin@example.fr
- Téléphone : +33 6 12 34 56 78
- Adresse : 15 Rue de la Paix, 75001 Paris


Stratégie : replace
Operators: {'DEFAULT': operator_name: replace, params: {}}

<LOCATION> du contact :
- Nom :<PERSON> :<EMAIL_ADDRESS> <ORGANIZATION> : <PHONE_NUMBER>
- Adresse :<LOCATION><LOCATION><LOCATION><LOCATION>
Anonymisations : 9
  - LOCATION             : '<LOCATION>' → 'e la Paix,' (op=replace)
  - LOCATION             : '<LOCATION>' → ': 15 Rue d' (op=replace)
  - LOCATION             : '<LOCATION>' → '- Adresse ' (op=replace)
  - LOCATION             : '<LOCATION>' → ' 34 56 78
' (op=replace)
  - PHONE_NUMBER         : '<PHONE_NUMBER>' → '.fr
- Téléphon' (op=replace)
  - ORGANIZATION         : '<ORGANIZATION>' → 'an.martin@exam' (op=replace)
  - EMAIL_ADDRESS        : '<EMAIL_ADDRESS>' → 'tin
- Email : j' (op=replace)
  - PERSON               : '<PERSON>' → ': Jean M' (op=replace)
  - LOCATION         

## 4. Tester les motifs personnalisés (numéro de bénévole / dossier)

La configuration d’exemple définit des **patterns personnalisés** pour :

- `NUMERO_BENEVOLE` (`BEN-123456`, `BENEVOLE 1234`, …) ;
- `NUMERO_DOSSIER` (`DOS-2026-0042`, `DOSSIER-123456`, …).

Voyons comment ils sont détectés et anonymisés dans un texte français.


In [ ]:
texte_custom = """
Le bénévole BEN-123456 a traité le dossier DOS-2026-0042.
Un autre bénévole BENEVOLE 7890 a également travaillé sur DOSSIER-567890.
"""

print("Texte avec motifs personnalisés :")
print(texte_custom)
print("\n--- Détections (fr) ---")

detections_custom = service.detect(texte_custom, language="fr")
for det in detections_custom:
    print(
        f"{det.entity_type:20s} | {det.text:30s} | score={det.score:.3f} | via={det.recognizer}"
    )

print("\n--- Anonymisation ---")
res_custom = service.anonymize(texte_custom, language="fr")
print(res_custom.text)
print(f"Anonymisations : {len(res_custom.items)}")

Texte avec motifs personnalisés :

Le bénévole BEN-123456 a traité le dossier DOS-2026-0042.
Un autre bénévole BENEVOLE 7890 a également travaillé sur DOSSIER-567890.


--- Détections (fr) ---
NUMERO_BENEVOLE      | BEN-123456                     | score=0.900 | via=numero_benevole
NUMERO_DOSSIER       | DOS-2026-0042                  | score=0.900 | via=numero_dossier
NUMERO_BENEVOLE      | BENEVOLE 7890                  | score=0.900 | via=numero_benevole
NUMERO_DOSSIER       | DOSSIER-567890                 | score=0.900 | via=numero_dossier
AGE                  |  BEN-123456                    | score=0.802 | via=AI4PrivacyRecognizer
US_SSN               |  DOSSIER-567890.
              | score=0.683 | via=AI4PrivacyRecognizer
LOCATION             |  7890                          | score=0.176 | via=AI4PrivacyRecognizer

--- Anonymisation ---
Operators: {'DEFAULT': operator_name: replace, params: {}}

Le bénévole<AGE> a traité le dossier <NUMERO_DOSSIER>.
Un autre bénévole <NUMERO_

## 5. Filtrer par type d’entité et par score de confiance

Deux usages fréquents :

- ne traiter **qu’un certain type d’entité** (ex. seulement les emails) ;
- ignorer les détections avec un **score de confiance trop faible**.

`PrivacyService.detect` et `PrivacyService.anonymize` exposent les paramètres :

- `entities=["EMAIL_ADDRESS", "PERSON", …]` ;
- `score_threshold=0.7` (par exemple).


In [6]:
texte_filtre = (
    "Contactez Jean Dupont par email à jean.dupont@example.fr ou au +33 6 12 34 56 78."
)

print("Texte :")
print(texte_filtre)

print("\n--- Détection uniquement des EMAIL_ADDRESS ---")
email_only = service.detect(texte_filtre, language="fr", entities=["EMAIL_ADDRESS"])
for det in email_only:
    print(f"{det.entity_type:20s} | {det.text:30s} | score={det.score:.3f}")

print("\n--- Détections avec score >= 0.7 ---")
for threshold in (None, 0.7, 0.9):
    label = "sans seuil" if threshold is None else f">= {threshold}"
    print(f"\nSeuil {label} :")
    dets = service.detect(texte_filtre, language="fr", score_threshold=threshold)
    for det in dets:
        print(f"{det.entity_type:20s} | {det.text:30s} | score={det.score:.3f}")

Texte :
Contactez Jean Dupont par email à jean.dupont@example.fr ou au +33 6 12 34 56 78.

--- Détection uniquement des EMAIL_ADDRESS ---
EMAIL_ADDRESS        | jean.dupont@example.fr         | score=1.000
EMAIL_ADDRESS        |  jean.dupont@example.fr        | score=0.017

--- Détections avec score >= 0.7 ---

Seuil sans seuil :
EMAIL_ADDRESS        | jean.dupont@example.fr         | score=1.000
PERSON               | Jean Dupont                    | score=0.850
URL                  | example.fr                     | score=0.500
PHONE_NUMBER         | +33 6 12 34 56 78              | score=0.400
EMAIL_ADDRESS        |  jean.dupont@example.fr        | score=0.017
PERSON               |  Jean                          | score=0.010

Seuil >= 0.7 :
EMAIL_ADDRESS        | jean.dupont@example.fr         | score=1.000
PERSON               | Jean Dupont                    | score=0.850

Seuil >= 0.9 :
EMAIL_ADDRESS        | jean.dupont@example.fr         | score=1.000


## 6. Résumé

Dans ce notebook, vous avez vu comment :

1. **Initialiser** `PrivacyService` avec une configuration orientée français ;
2. **Détecter** des PII dans des textes français simples ;
3. **Anonymiser** ces PII avec différentes stratégies ;
4. Utiliser des **motifs personnalisés** pour les numéros de bénévoles / dossiers ;
5. **Filtrer** par type d’entité et par score de confiance.

Pour aller plus loin :

- adaptez `config.yaml` à vos propres besoins (entités, stratégies, seuils) ;
- injectez vos textes réels (comptes rendus, emails, exports CSV, etc.) ;
- combinez cette logique dans vos scripts ou API métier.
